In [2]:
"""
Streamlined Roof Texture Classification Training Script
Clean, efficient implementation with minimal redundancy - LIBPNG WARNING COMPLETELY SUPPRESSED
"""

import os
import pickle
import warnings
import sys
import subprocess
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from multiprocessing import cpu_count

# Import additional required libraries for k-fold validation
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import pandas as pd

# NUCLEAR OPTION: Complete system-level warning suppression
os.environ['OPENCV_LOG_LEVEL'] = 'SILENT'
os.environ['OPENCV_OPENCL_DEVICE'] = 'disabled'
os.environ['PYTHONWARNINGS'] = 'ignore'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '0'
os.environ['OPENCV_IO_ENABLE_JASPER'] = '0'

# Redirect stderr at the file descriptor level (lowest level possible)
import os
import sys

# Save original stderr file descriptor
original_stderr_fd = os.dup(2)

def silence_stderr():
    """Redirect stderr to null at the file descriptor level"""
    null_fd = os.open(os.devnull, os.O_WRONLY)
    os.dup2(null_fd, 2)
    os.close(null_fd)

def restore_stderr():
    """Restore original stderr"""
    os.dup2(original_stderr_fd, 2)

# Apply maximum Python-level warning suppression
warnings.filterwarnings("ignore")
warnings.simplefilter("ignore")
for category in [UserWarning, DeprecationWarning, RuntimeWarning, FutureWarning, Warning]:
    warnings.filterwarnings("ignore", category=category)

# Specific libpng warning patterns
libpng_patterns = [".*iCCP.*", ".*libpng.*", ".*sRGB.*", ".*profile.*", ".*PNG.*", ".*color.*"]
for pattern in libpng_patterns:
    warnings.filterwarnings("ignore", message=pattern)

# Comprehensive logging suppression
import logging
logging.disable(logging.CRITICAL)
for logger_name in ['PIL', 'PIL.PngImagePlugin', 'PIL.Image', 'matplotlib', 'cv2']:
    logging.getLogger(logger_name).setLevel(logging.CRITICAL)
    logging.getLogger(logger_name).disabled = True

# Context manager that silences everything
import contextlib

@contextlib.contextmanager
def nuclear_silence():
    """Ultimate silence - redirects stderr at file descriptor level"""
    silence_stderr()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            yield
        finally:
            restore_stderr()

# Import libraries with maximum suppression
with nuclear_silence():
    import PIL.Image
    PIL.Image.warnings.simplefilter('ignore')
    from PIL import Image
    import cv2
    import numpy as np
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import DataLoader as TorchDataLoader, Dataset
    from torchvision import models, transforms
    import matplotlib.pyplot as plt
    from tqdm import tqdm
    from sklearn.model_selection import train_test_split
    from sklearn.utils.class_weight import compute_class_weight

# Additional suppression for OpenCV warnings
os.environ['OPENCV_LOG_LEVEL'] = 'ERROR'


class Config:
    """Configuration class for training parameters"""
    BASE_PATH = "/home/student/sky-scan/data"
    BATCH_SIZE = 16
    NUM_EPOCHS = 150
    LEARNING_RATE = 0.001
    IMAGE_SIZE = 224
    NUM_WORKERS = 2
    PATIENCE = 5
    MIN_CONTOUR_AREA = 100
    MODEL_SAVE_PATH = "models/roof_texture_model.pth"


class KFoldConfig(Config):
    """Extended configuration for K-Fold validation"""
    K_FOLDS = 5
    NUM_EPOCHS = 50  # Reduced epochs per fold for faster training
    MODEL_SAVE_DIR = "models/kfold"


def set_seed(seed=42):
    """Set random seeds for reproducibility"""
    torch.manual_seed(seed)
    np.random.seed(seed)


def get_device():
    """Get available device"""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if torch.cuda.is_available():
        torch.backends.cudnn.benchmark = True
        print(f"Using GPU: {torch.cuda.get_device_name()}")
    return device


class RoofDataLoader:
    """Handles data loading and preprocessing"""

    def __init__(self, base_path):
        self.original_dir = f"{base_path}/patch"
        self.binary_dir = f"{base_path}/patch-binary"
        self.color_dir = f"{base_path}/patch-texture"
        self.cache_dir = f"{base_path}/cache"
        os.makedirs(self.cache_dir, exist_ok=True)

    def load_file_paths(self):
        """Load file paths with caching"""
        cache_file = os.path.join(self.cache_dir, 'file_paths.pkl')

        if os.path.exists(cache_file):
            print("Loading paths from cache...")
            with open(cache_file, 'rb') as f:
                return pickle.load(f)

        print("Scanning directories...")
        filenames = sorted(os.listdir(self.original_dir))

        def check_files(filename):
            paths = [
                os.path.join(self.original_dir, filename),
                os.path.join(self.binary_dir, filename),
                os.path.join(self.color_dir, filename)
            ]
            return paths if all(os.path.exists(p) for p in paths) else None

        with ThreadPoolExecutor(max_workers=cpu_count()) as executor:
            results = list(tqdm(
                executor.map(check_files, filenames),
                total=len(filenames),
                desc="Checking files"
            ))

        valid_paths = [r for r in results if r is not None]
        paths = list(zip(*valid_paths)) if valid_paths else ([], [], [])

        with open(cache_file, 'wb') as f:
            pickle.dump(paths, f)

        return paths


def get_label_from_color(color_patch, tolerance=15):
    """Extract label from color-coded patch"""
    colors = {
        "smooth": np.array([83, 217, 56]),
        "average": np.array([252, 240, 3]),
        "rough": np.array([255, 0, 0])
    }

    pixels = color_patch.reshape(-1, 3)
    counts = {}

    for label, color in colors.items():
        distances = np.abs(pixels - color).max(axis=1)
        counts[label] = np.sum(distances <= tolerance)

    # Return label with highest count
    return max(counts, key=counts.get) if max(counts.values()) > 50 else None


def process_image(args):
    """Process a single image to extract patches and labels"""
    img_path, binary_path, color_path, min_area, target_size = args

    try:
        # Load images with NUCLEAR warning suppression
        with nuclear_silence():
            img = cv2.imread(img_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0

            binary = cv2.imread(binary_path, cv2.IMREAD_GRAYSCALE)
            color = cv2.imread(color_path)
            color = cv2.cvtColor(color, cv2.COLOR_BGR2RGB)

        # Find contours
        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        valid_contours = [c for c in contours if cv2.contourArea(c) > min_area]

        patches = []
        labels = []

        if valid_contours:
            for contour in valid_contours:
                x, y, w, h = cv2.boundingRect(contour)

                # Add padding
                padding = 10
                x = max(0, x - padding)
                y = max(0, y - padding)
                w = min(img.shape[1] - x, w + 2 * padding)
                h = min(img.shape[0] - y, h + 2 * padding)

                if w < 20 or h < 20:
                    continue

                # Extract and resize patches
                img_patch = cv2.resize(img[y:y+h, x:x+w], target_size)
                color_patch = cv2.resize(color[y:y+h, x:x+w], target_size)

                label = get_label_from_color(color_patch)

                if label is None:
                    continue  # Skip if no valid label

                patches.append(img_patch)
                labels.append(label)
        else:
            # Use full image if no contours
            img_patch = cv2.resize(img, target_size)
            color_patch = cv2.resize(color, target_size)
            label = get_label_from_color(color_patch)

            if label is not None:
                patches.append(img_patch)
                labels.append(label)

        return patches, labels

    except Exception as e:
        print(f"Error processing image: {e}")
        return [], []


def extract_patches(image_paths, binary_paths, color_paths, config):
    """Extract patches from all images"""
    cache_file = os.path.join(config.BASE_PATH, 'cache', 'patches_no_contour_removed.pkl')

    if os.path.exists(cache_file):
        print("Loading patches from cache...")
        with open(cache_file, 'rb') as f:
            return pickle.load(f)

    print("Extracting patches...")
    args_list = [
        (image_paths[i], binary_paths[i], color_paths[i],
         config.MIN_CONTOUR_AREA, (config.IMAGE_SIZE, config.IMAGE_SIZE))
        for i in range(len(image_paths))
    ]

    all_patches = []
    all_labels = []

    with ThreadPoolExecutor(max_workers=cpu_count()) as executor:
        results = list(tqdm(
            executor.map(process_image, args_list),
            total=len(args_list),
            desc="Processing images"
        ))

    for patches, labels in results:
        all_patches.extend(patches)
        all_labels.extend(labels)

    # Filter out any remaining None or "no_contour" labels
    filtered_patches = []
    filtered_labels = []
    for patch, label in zip(all_patches, all_labels):
        if label is not None and label != "no_contour":
            filtered_patches.append(patch)
            filtered_labels.append(label)

    patches_array = np.array(filtered_patches) if filtered_patches else np.array([])

    with open(cache_file, 'wb') as f:
        pickle.dump((patches_array, filtered_labels), f)

    return patches_array, filtered_labels


def balance_dataset(patches, labels, strategy='undersample', max_samples_per_class=None):
    """
    Balance dataset using different strategies

    Args:
        patches: numpy array of image patches
        labels: list of labels
        strategy: 'undersample', 'oversample', or 'hybrid'
        max_samples_per_class: maximum samples per class (for undersampling)
    """
    from collections import Counter
    import numpy as np

    print(f"\n=== DATASET BALANCING ({strategy.upper()}) ===")

    # Count original distribution
    label_counts = Counter(labels)
    print("Original distribution:")
    for label, count in sorted(label_counts.items()):
        print(f"  {label}: {count} samples ({count/len(labels)*100:.1f}%)")

    if strategy == 'undersample':
        # Undersample majority classes
        if max_samples_per_class is None:
            max_samples_per_class = min(label_counts.values())

        print(f"\nUndersampling to max {max_samples_per_class} samples per class...")

        balanced_patches = []
        balanced_labels = []

        # Group by labels
        label_to_indices = {}
        for idx, label in enumerate(labels):
            if label not in label_to_indices:
                label_to_indices[label] = []
            label_to_indices[label].append(idx)

        # Sample from each class
        for label, indices in label_to_indices.items():
            if len(indices) > max_samples_per_class:
                # Random sample
                np.random.seed(42)
                selected_indices = np.random.choice(indices, max_samples_per_class, replace=False)
            else:
                selected_indices = indices

            for idx in selected_indices:
                balanced_patches.append(patches[idx])
                balanced_labels.append(labels[idx])

        balanced_patches = np.array(balanced_patches)
        balanced_labels = balanced_labels

    elif strategy == 'oversample':
        # Oversample minority classes
        max_count = max(label_counts.values())
        print(f"\nOversampling to {max_count} samples per class...")

        balanced_patches = []
        balanced_labels = []

        # Group by labels
        label_to_data = {}
        for idx, label in enumerate(labels):
            if label not in label_to_data:
                label_to_data[label] = []
            label_to_data[label].append(patches[idx])

        # Oversample each class
        for label, class_patches in label_to_data.items():
            current_count = len(class_patches)
            needed = max_count

            # Add original samples
            for patch in class_patches:
                balanced_patches.append(patch)
                balanced_labels.append(label)

            # Add oversampled data
            if current_count < needed:
                np.random.seed(42)
                oversample_indices = np.random.choice(current_count, needed - current_count, replace=True)
                for idx in oversample_indices:
                    balanced_patches.append(class_patches[idx])
                    balanced_labels.append(label)

        balanced_patches = np.array(balanced_patches)

    elif strategy == 'hybrid':
        # Hybrid: Cap high classes and boost low classes
        median_count = int(np.median(list(label_counts.values())))
        target_count = median_count * 2  # Reasonable target

        print(f"\nHybrid balancing to ~{target_count} samples per class...")

        balanced_patches = []
        balanced_labels = []

        # Group by labels
        label_to_data = {}
        for idx, label in enumerate(labels):
            if label not in label_to_data:
                label_to_data[label] = []
            label_to_data[label].append(patches[idx])

        # Balance each class
        for label, class_patches in label_to_data.items():
            current_count = len(class_patches)

            if current_count > target_count:
                # Undersample
                np.random.seed(42)
                selected_indices = np.random.choice(current_count, target_count, replace=False)
                for idx in selected_indices:
                    balanced_patches.append(class_patches[idx])
                    balanced_labels.append(label)
            elif current_count < target_count:
                # Oversample
                for patch in class_patches:
                    balanced_patches.append(patch)
                    balanced_labels.append(label)

                needed = target_count - current_count
                np.random.seed(42)
                oversample_indices = np.random.choice(current_count, needed, replace=True)
                for idx in oversample_indices:
                    balanced_patches.append(class_patches[idx])
                    balanced_labels.append(label)
            else:
                # Keep as is
                for patch in class_patches:
                    balanced_patches.append(patch)
                    balanced_labels.append(label)

        balanced_patches = np.array(balanced_patches)

    # Print final distribution
    final_counts = Counter(balanced_labels)
    print("\nFinal distribution:")
    for label, count in sorted(final_counts.items()):
        print(f"  {label}: {count} samples ({count/len(balanced_labels)*100:.1f}%)")

    return balanced_patches, balanced_labels


class RoofDataset(Dataset):
    """PyTorch dataset for roof patches"""

    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

        # Create label mapping
        unique_labels = sorted(list(set(labels)))
        self.label_to_idx = {label: idx for idx, label in enumerate(unique_labels)}
        self.idx_to_label = {idx: label for label, idx in self.label_to_idx.items()}

        # Convert labels to indices
        self.label_indices = [self.label_to_idx[label] for label in labels]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        if image.dtype != np.uint8:
            image = (image * 255).astype(np.uint8)

        if self.transform:
            with nuclear_silence():  # Suppress warnings during transform pipeline
                image = self.transform(image)

        return image, self.label_indices[idx]


class RoofTextureClassifier(nn.Module):
    """Simplified CNN model for roof texture classification"""

    def __init__(self, num_classes):
        super().__init__()

        # Use EfficientNet-B0 backbone
        self.backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

        # Replace classifier
        num_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(num_features, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.backbone(x)


class Trainer:
    """Model trainer class"""

    def __init__(self, model, device, config):
        self.model = model
        self.device = device
        self.config = config

    def train_epoch(self, train_loader, optimizer, criterion):
        """Train for one epoch"""
        self.model.train()
        total_loss = 0
        correct = 0
        total = 0

        for inputs, targets in tqdm(train_loader, desc="Training"):
            with nuclear_silence():  # Suppress warnings during data loading
                inputs, targets = inputs.to(self.device), targets.to(self.device)

            optimizer.zero_grad()
            outputs = self.model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

        return total_loss / len(train_loader), 100. * correct / total

    def validate(self, val_loader, criterion):
        """Validate model"""
        self.model.eval()
        total_loss = 0
        correct = 0
        total = 0

        with torch.no_grad():
            for inputs, targets in val_loader:
                with nuclear_silence():  # Suppress warnings during data loading
                    inputs, targets = inputs.to(self.device), targets.to(self.device)
                outputs = self.model(inputs)
                loss = criterion(outputs, targets)

                total_loss += loss.item()
                _, predicted = outputs.max(1)
                total += targets.size(0)
                correct += predicted.eq(targets).sum().item()

        return total_loss / len(val_loader), 100. * correct / total

    def train(self, train_loader, val_loader, class_weights):
        """Full training loop"""
        criterion = nn.CrossEntropyLoss(weight=class_weights)
        optimizer = optim.Adam(self.model.parameters(), lr=self.config.LEARNING_RATE)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3)

        best_val_acc = 0
        patience_counter = 0

        train_losses, val_losses = [], []
        train_accs, val_accs = [], []

        for epoch in range(self.config.NUM_EPOCHS):
            train_loss, train_acc = self.train_epoch(train_loader, optimizer, criterion)
            val_loss, val_acc = self.validate(val_loader, criterion)

            scheduler.step(val_loss)

            train_losses.append(train_loss)
            val_losses.append(val_loss)
            train_accs.append(train_acc)
            val_accs.append(val_acc)

            print(f"Epoch {epoch+1}/{self.config.NUM_EPOCHS}: "
                  f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.2f}% | "
                  f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.2f}%")

            # Save best model
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                patience_counter = 0
                self.save_model()
                print(f"New best model saved! Val Acc: {val_acc:.2f}%")
            else:
                patience_counter += 1

            if patience_counter >= self.config.PATIENCE:
                print("Early stopping!")
                break

        return {
            'train_losses': train_losses,
            'val_losses': val_losses,
            'train_accs': train_accs,
            'val_accs': val_accs,
            'best_val_acc': best_val_acc
        }

    def save_model(self):
        """Save model checkpoint"""
        os.makedirs(os.path.dirname(self.config.MODEL_SAVE_PATH), exist_ok=True)
        torch.save({
            'model_state_dict': self.model.state_dict(),
            'model_class': type(self.model).__name__
        }, self.config.MODEL_SAVE_PATH)


class KFoldTrainer:
    """Enhanced trainer class for K-Fold validation"""

    def __init__(self, device, config):
        self.device = device
        self.config = config
        self.fold_results = []

    def create_model(self, num_classes):
        """Create a fresh model instance"""
        model = RoofTextureClassifier(num_classes).to(self.device)
        return model

    def train_fold(self, model, train_loader, val_loader, class_weights, fold_num):
        """Train model for one fold"""
        print(f"\n{'='*50}")
        print(f"🔄 Training Fold {fold_num}/{self.config.K_FOLDS}")
        print(f"{'='*50}")

        criterion = nn.CrossEntropyLoss(weight=class_weights)
        optimizer = optim.Adam(model.parameters(), lr=self.config.LEARNING_RATE)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3)

        best_val_acc = 0
        patience_counter = 0
        fold_history = {
            'train_losses': [], 'val_losses': [],
            'train_accs': [], 'val_accs': []
        }

        # Training progress bar for this fold
        epoch_pbar = tqdm(range(self.config.NUM_EPOCHS),
                         desc=f"🎯 Fold {fold_num} Progress",
                         ncols=120)

        for epoch in epoch_pbar:
            # Train epoch
            train_loss, train_acc = self.train_epoch(model, train_loader, optimizer, criterion)

            # Validate epoch
            val_loss, val_acc = self.validate(model, val_loader, criterion)

            scheduler.step(val_loss)

            fold_history['train_losses'].append(train_loss)
            fold_history['val_losses'].append(val_loss)
            fold_history['train_accs'].append(train_acc)
            fold_history['val_accs'].append(val_acc)

            # Update progress bar
            epoch_pbar.set_postfix({
                'Train Acc': f'{train_acc:.2f}%',
                'Val Acc': f'{val_acc:.2f}%',
                'Best': f'{best_val_acc:.2f}%'
            })

            # Early stopping
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                patience_counter = 0
                self.save_fold_model(model, fold_num)
            else:
                patience_counter += 1

            if patience_counter >= self.config.PATIENCE:
                print(f"⏹️  Early stopping at epoch {epoch+1}")
                break

        epoch_pbar.close()
        fold_history['best_val_acc'] = best_val_acc

        print(f"✅ Fold {fold_num} completed - Best Val Acc: {best_val_acc:.2f}%")
        return fold_history

    def train_epoch(self, model, train_loader, optimizer, criterion):
        """Train for one epoch"""
        model.train()
        total_loss = 0
        correct = 0
        total = 0

        for inputs, targets in train_loader:
            with nuclear_silence():
                inputs, targets = inputs.to(self.device), targets.to(self.device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

        return total_loss / len(train_loader), 100. * correct / total

    def validate(self, model, val_loader, criterion):
        """Validate model"""
        model.eval()
        total_loss = 0
        correct = 0
        total = 0
        all_predictions = []
        all_targets = []

        with torch.no_grad():
            for inputs, targets in val_loader:
                with nuclear_silence():
                    inputs, targets = inputs.to(self.device), targets.to(self.device)
                outputs = model(inputs)
                loss = criterion(outputs, targets)

                total_loss += loss.item()
                _, predicted = outputs.max(1)
                total += targets.size(0)
                correct += predicted.eq(targets).sum().item()

                all_predictions.extend(predicted.cpu().numpy())
                all_targets.extend(targets.cpu().numpy())

        return total_loss / len(val_loader), 100. * correct / total

    def save_fold_model(self, model, fold_num):
        """Save model for specific fold"""
        os.makedirs(self.config.MODEL_SAVE_DIR, exist_ok=True)
        model_path = os.path.join(self.config.MODEL_SAVE_DIR, f'fold_{fold_num}_best.pth')
        torch.save({
            'model_state_dict': model.state_dict(),
            'fold': fold_num,
            'model_class': type(model).__name__
        }, model_path)

    def run_kfold_validation(self, patches, labels):
        """Run complete K-Fold cross validation"""
        print(f"\n🚀 Starting {self.config.K_FOLDS}-Fold Cross Validation")
        print(f"📊 Total samples: {len(patches)}")
        print(f"🏷️  Classes: {sorted(list(set(labels)))}")

        # Convert labels to indices for stratification
        label_to_idx = {label: idx for idx, label in enumerate(sorted(list(set(labels))))}
        label_indices = [label_to_idx[label] for label in labels]

        # Initialize K-Fold
        skf = StratifiedKFold(n_splits=self.config.K_FOLDS, shuffle=True, random_state=42)

        fold_results = []
        all_fold_histories = []

        # Get transforms
        train_transform, val_transform = get_transforms(self.config.IMAGE_SIZE)

        for fold, (train_idx, val_idx) in enumerate(skf.split(patches, label_indices), 1):
            print(f"\n📁 Preparing Fold {fold}/{self.config.K_FOLDS}")

            # Split data for this fold
            X_train_fold = patches[train_idx]
            X_val_fold = patches[val_idx]
            y_train_fold = [labels[i] for i in train_idx]
            y_val_fold = [labels[i] for i in val_idx]

            print(f"   📈 Training samples: {len(X_train_fold)}")
            print(f"   📊 Validation samples: {len(X_val_fold)}")

            # Create datasets for this fold
            train_dataset = RoofDataset(X_train_fold, y_train_fold, train_transform)
            val_dataset = RoofDataset(X_val_fold, y_val_fold, val_transform)

            # Create data loaders
            train_loader = TorchDataLoader(
                train_dataset, batch_size=self.config.BATCH_SIZE,
                shuffle=True, num_workers=self.config.NUM_WORKERS
            )
            val_loader = TorchDataLoader(
                val_dataset, batch_size=self.config.BATCH_SIZE,
                shuffle=False, num_workers=self.config.NUM_WORKERS
            )

            # Create fresh model for this fold
            num_classes = len(train_dataset.label_to_idx)
            model = self.create_model(num_classes)

            # Calculate class weights for this fold
            y_indices_fold = [train_dataset.label_to_idx[label] for label in y_train_fold]
            class_weights = compute_class_weight('balanced',
                                               classes=np.unique(y_indices_fold),
                                               y=y_indices_fold)
            class_weights = torch.FloatTensor(class_weights).to(self.device)

            # Train this fold
            fold_history = self.train_fold(model, train_loader, val_loader, class_weights, fold)
            all_fold_histories.append(fold_history)

            # Evaluate this fold
            best_model_path = os.path.join(self.config.MODEL_SAVE_DIR, f'fold_{fold}_best.pth')
            model.load_state_dict(torch.load(best_model_path)['model_state_dict'])

            # Final evaluation on validation set
            val_loss, val_acc = self.validate(model, val_loader, nn.CrossEntropyLoss())

            fold_result = {
                'fold': fold,
                'val_accuracy': val_acc,
                'val_loss': val_loss,
                'train_samples': len(X_train_fold),
                'val_samples': len(X_val_fold),
                'history': fold_history
            }
            fold_results.append(fold_result)

            print(f"✅ Fold {fold} Final Results:")
            print(f"   🎯 Validation Accuracy: {val_acc:.2f}%")
            print(f"   📉 Validation Loss: {val_loss:.4f}")

        # Calculate overall statistics
        val_accuracies = [result['val_accuracy'] for result in fold_results]
        mean_acc = np.mean(val_accuracies)
        std_acc = np.std(val_accuracies)

        print(f"\n🏆 K-FOLD CROSS VALIDATION RESULTS")
        print(f"{'='*60}")
        for i, result in enumerate(fold_results, 1):
            print(f"Fold {i}: {result['val_accuracy']:.2f}%")
        print(f"{'='*60}")
        print(f"📊 Mean Accuracy: {mean_acc:.2f} ± {std_acc:.2f}%")
        print(f"🔝 Best Fold: {max(val_accuracies):.2f}%")
        print(f"📉 Worst Fold: {min(val_accuracies):.2f}%")

        return {
            'fold_results': fold_results,
            'mean_accuracy': mean_acc,
            'std_accuracy': std_acc,
            'best_accuracy': max(val_accuracies),
            'worst_accuracy': min(val_accuracies),
            'all_histories': all_fold_histories
        }


def get_transforms(image_size):
    """Get data transforms"""
    train_transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((image_size, image_size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    val_transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    return train_transform, val_transform


def plot_training_history(history):
    """Plot training history"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(history['train_losses'], label='Train Loss')
    ax1.plot(history['val_losses'], label='Val Loss')
    ax1.set_title('Training Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True)

    ax2.plot(history['train_accs'], label='Train Acc')
    ax2.plot(history['val_accs'], label='Val Acc')
    ax2.set_title('Training Accuracy')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy (%)')
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.show()


def plot_kfold_results(kfold_results):
    """Plot comprehensive K-Fold validation results"""
    fold_results = kfold_results['fold_results']
    all_histories = kfold_results['all_histories']

    # Create a comprehensive visualization
    fig = plt.figure(figsize=(20, 12))

    # 1. Fold-wise accuracy comparison
    plt.subplot(2, 4, 1)
    fold_nums = [result['fold'] for result in fold_results]
    accuracies = [result['val_accuracy'] for result in fold_results]

    bars = plt.bar(fold_nums, accuracies, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd'][:len(fold_nums)])
    plt.axhline(y=kfold_results['mean_accuracy'], color='red', linestyle='--',
                label=f'Mean: {kfold_results["mean_accuracy"]:.2f}%')
    plt.title('📊 Validation Accuracy per Fold')
    plt.xlabel('Fold')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.grid(True, alpha=0.3)

    # Add value labels on bars
    for bar, acc in zip(bars, accuracies):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{acc:.1f}%', ha='center', va='bottom')

    # 2. Training curves for all folds
    plt.subplot(2, 4, 2)
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
    for i, history in enumerate(all_histories):
        plt.plot(history['train_accs'], color=colors[i % len(colors)], alpha=0.7,
                linestyle='-', label=f'Fold {i+1} Train')
        plt.plot(history['val_accs'], color=colors[i % len(colors)], alpha=0.9,
                linestyle='--', label=f'Fold {i+1} Val')
    plt.title('🎯 Training Curves - All Folds')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)

    # 3. Loss curves for all folds
    plt.subplot(2, 4, 3)
    for i, history in enumerate(all_histories):
        plt.plot(history['train_losses'], color=colors[i % len(colors)], alpha=0.7,
                linestyle='-', label=f'Fold {i+1} Train')
        plt.plot(history['val_losses'], color=colors[i % len(colors)], alpha=0.9,
                linestyle='--', label=f'Fold {i+1} Val')
    plt.title('📉 Loss Curves - All Folds')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)

    # 4. Accuracy distribution
    plt.subplot(2, 4, 4)
    plt.hist(accuracies, bins=max(3, len(accuracies)//2), alpha=0.7, color='skyblue', edgecolor='black')
    plt.axvline(kfold_results['mean_accuracy'], color='red', linestyle='--',
                label=f'Mean: {kfold_results["mean_accuracy"]:.2f}%')
    plt.axvline(kfold_results['best_accuracy'], color='green', linestyle='--',
                label=f'Best: {kfold_results["best_accuracy"]:.2f}%')
    plt.title('📈 Accuracy Distribution')
    plt.xlabel('Accuracy (%)')
    plt.ylabel('Frequency')
    plt.legend()
    plt.grid(True, alpha=0.3)

    # 5. Average training curves across all folds
    plt.subplot(2, 4, 5)
    # Calculate mean and std for each epoch
    max_epochs = max(len(hist['train_accs']) for hist in all_histories)
    mean_train_accs = []
    std_train_accs = []
    mean_val_accs = []
    std_val_accs = []

    for epoch in range(max_epochs):
        train_accs_epoch = [hist['train_accs'][epoch] for hist in all_histories if epoch < len(hist['train_accs'])]
        val_accs_epoch = [hist['val_accs'][epoch] for hist in all_histories if epoch < len(hist['val_accs'])]

        if train_accs_epoch:
            mean_train_accs.append(np.mean(train_accs_epoch))
            std_train_accs.append(np.std(train_accs_epoch))
        if val_accs_epoch:
            mean_val_accs.append(np.mean(val_accs_epoch))
            std_val_accs.append(np.std(val_accs_epoch))

    epochs = range(len(mean_train_accs))
    plt.plot(epochs, mean_train_accs, 'b-', label='Mean Train Acc', linewidth=2)
    plt.fill_between(epochs,
                     np.array(mean_train_accs) - np.array(std_train_accs),
                     np.array(mean_train_accs) + np.array(std_train_accs),
                     alpha=0.2, color='blue')

    epochs_val = range(len(mean_val_accs))
    plt.plot(epochs_val, mean_val_accs, 'r--', label='Mean Val Acc', linewidth=2)
    plt.fill_between(epochs_val,
                     np.array(mean_val_accs) - np.array(std_val_accs),
                     np.array(mean_val_accs) + np.array(std_val_accs),
                     alpha=0.2, color='red')

    plt.title('📊 Average Training Curves (±1 std)')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.grid(True, alpha=0.3)

    # 6. Summary statistics
    plt.subplot(2, 4, 6)
    plt.axis('off')
    stats_text = f"""
    🏆 K-Fold Validation Summary

    📊 Number of Folds: {kfold_results['fold_results'][0]['fold']} - {fold_results[-1]['fold']}

    📈 Mean Accuracy: {kfold_results['mean_accuracy']:.2f}%
    📏 Std Deviation: ±{kfold_results['std_accuracy']:.2f}%

    🔝 Best Fold: {kfold_results['best_accuracy']:.2f}%
    📉 Worst Fold: {kfold_results['worst_accuracy']:.2f}%

    📊 Accuracy Range: {kfold_results['best_accuracy'] - kfold_results['worst_accuracy']:.2f}%

    🎯 Model Stability: {'High' if kfold_results['std_accuracy'] < 2.0 else 'Medium' if kfold_results['std_accuracy'] < 5.0 else 'Low'}
    """
    plt.text(0.1, 0.9, stats_text, fontsize=12, verticalalignment='top',
             bbox=dict(boxstyle="round,pad=0.3", facecolor="lightblue", alpha=0.7))

    # 7. Box plot of accuracies
    plt.subplot(2, 4, 7)
    plt.boxplot(accuracies, patch_artist=True,
                boxprops=dict(facecolor='lightblue', alpha=0.7),
                medianprops=dict(color='red', linewidth=2))
    plt.ylabel('Accuracy (%)')
    plt.title('📦 Accuracy Distribution\n(Box Plot)')
    plt.grid(True, alpha=0.3)

    # 8. Sample distribution per fold
    plt.subplot(2, 4, 8)
    train_samples = [result['train_samples'] for result in fold_results]
    val_samples = [result['val_samples'] for result in fold_results]

    x = np.arange(len(fold_nums))
    width = 0.35

    plt.bar(x - width/2, train_samples, width, label='Training', alpha=0.7)
    plt.bar(x + width/2, val_samples, width, label='Validation', alpha=0.7)

    plt.xlabel('Fold')
    plt.ylabel('Number of Samples')
    plt.title('📊 Sample Distribution per Fold')
    plt.xticks(x, fold_nums)
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


def save_kfold_results(kfold_results, config):
    """Save K-Fold results to file"""
    results_file = os.path.join(config.MODEL_SAVE_DIR, 'kfold_results.pkl')
    with open(results_file, 'wb') as f:
        pickle.dump(kfold_results, f)

    # Also save as readable text
    text_file = os.path.join(config.MODEL_SAVE_DIR, 'kfold_summary.txt')
    with open(text_file, 'w') as f:
        f.write("K-FOLD CROSS VALIDATION RESULTS\n")
        f.write("="*50 + "\n\n")

        for result in kfold_results['fold_results']:
            f.write(f"Fold {result['fold']}:\n")
            f.write(f"  Validation Accuracy: {result['val_accuracy']:.2f}%\n")
            f.write(f"  Validation Loss: {result['val_loss']:.4f}\n")
            f.write(f"  Training Samples: {result['train_samples']}\n")
            f.write(f"  Validation Samples: {result['val_samples']}\n\n")

        f.write("OVERALL STATISTICS:\n")
        f.write(f"Mean Accuracy: {kfold_results['mean_accuracy']:.2f} ± {kfold_results['std_accuracy']:.2f}%\n")
        f.write(f"Best Accuracy: {kfold_results['best_accuracy']:.2f}%\n")
        f.write(f"Worst Accuracy: {kfold_results['worst_accuracy']:.2f}%\n")

    print(f"📁 Results saved to: {config.MODEL_SAVE_DIR}")



In [3]:
print("🚀 STARTING K-FOLD CROSS VALIDATION")
print("="*60)

# Initialize K-Fold configuration
kfold_config = KFoldConfig()
set_seed(42)
device = get_device()

# Load and prepare data (same as before)
data_loader = RoofDataLoader(kfold_config.BASE_PATH)
image_paths, binary_paths, color_paths = data_loader.load_file_paths()
print(f"Found {len(image_paths)} image sets")

# Extract patches
patches, labels = extract_patches(image_paths, binary_paths, color_paths, kfold_config)
print(f"Extracted {len(patches)} patches")

# Show original distribution
label_counts = Counter(labels)
print("Original label distribution:", label_counts)

# Apply class balancing
patches, labels = balance_dataset(patches, labels, strategy='hybrid')

print(f"\n🎯 Final dataset: {len(patches)} patches, {len(set(labels))} classes")
print(f"📊 Classes: {sorted(list(set(labels)))}")


🚀 STARTING K-FOLD CROSS VALIDATION
Using GPU: Tesla V100-PCIE-16GB
Loading paths from cache...
Found 16200 image sets
Loading patches from cache...
Extracted 23326 patches
Original label distribution: Counter({'average': 10976, 'rough': 6519, 'smooth': 5831})

=== DATASET BALANCING (HYBRID) ===
Original distribution:
  average: 10976 samples (47.1%)
  rough: 6519 samples (27.9%)
  smooth: 5831 samples (25.0%)

Hybrid balancing to ~13038 samples per class...

Final distribution:
  average: 13038 samples (33.3%)
  rough: 13038 samples (33.3%)
  smooth: 13038 samples (33.3%)

🎯 Final dataset: 39114 patches, 3 classes
📊 Classes: ['average', 'rough', 'smooth']


In [ ]:
# Run K-Fold Cross Validation
kfold_trainer = KFoldTrainer(device, kfold_config)
kfold_results = kfold_trainer.run_kfold_validation(patches, labels)

# Plot comprehensive results
plot_kfold_results(kfold_results)

# Save results
save_kfold_results(kfold_results, kfold_config)

print("\n🎉 K-FOLD VALIDATION COMPLETED!")
print("="*60)



🚀 Starting 5-Fold Cross Validation
📊 Total samples: 39114
🏷️  Classes: ['average', 'rough', 'smooth']

📁 Preparing Fold 1/5
